# ColliderFM Model, Training, And SSL Walkthrough

If you're trying to get oriented in the model code, this is the notebook to use.

It follows the same path as the training script and ties the moving pieces together:

- the point-view contract the model actually sees
- the augmentations we currently use
- how teacher and student batches are built
- what the model returns at point level and event level
- how the current SSL losses are formed
- what plots are worth checking when we want to know whether training is learning anything useful


## Where This Fits

The dataset notebook stays on the raw-data side of the repo.

This one starts at the first model-facing representation and follows the same path as `scripts/train.py`:

1. build a `PointView`
2. make augmented teacher and student views
3. batch them into a `DistillationBatch`
4. run the student/teacher model
5. inspect the SSL losses and training curves

The goal is not to make the notebook read like a lecture. It is meant to be the notebook you'd hand to a new teammate so they can get productive quickly.


In [ ]:
import inspect
import json
import sys
from collections.abc import Sequence
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from collider_fm.diagnostics import compute_pca, encode_view, load_checkpoint, load_events, tensor_summary, to_numpy
from collider_fm.model import PandaSelfDistillation, create_small_panda_model, create_training_panda_model, pointwise_panda_loss
from collider_fm.project_config import load_project_config, model_factory_kwargs, to_plain_container
from collider_fm.views import augment_point_view, batch_point_views, build_distillation_views, build_point_view_from_event

plt.style.use('default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = False


## Configuration

The notebook can run with a fresh model or with a saved checkpoint.

If a checkpoint is available, we also load the matching `metrics.jsonl` file so the model internals and the training curves live in the same walkthrough.


In [ ]:
SEED = 7
PROJECT_CONFIG = to_plain_container(load_project_config())
DATA_CONFIG = PROJECT_CONFIG['data']
VIEW_CONFIG = PROJECT_CONFIG['views']
MODEL_CONFIG = PROJECT_CONFIG['model']
DIAGNOSTICS_CONFIG = PROJECT_CONFIG['diagnostics']

DETAIL_SPLIT = DIAGNOSTICS_CONFIG['detail_split']
REPRESENTATION_SPLIT = DIAGNOSTICS_CONFIG['representation_split']
TRAIN_VIEW_EVENT_COUNT = 2
DATASET_NAME = DATA_CONFIG['dataset_name']
DATASET_TYPE = DATA_CONFIG['dataset_type']
PU_CONFIG = DATA_CONFIG['pu_config']
CACHE_DIR = DATA_CONFIG['cache_dir']
DATASET_REVISION = DATA_CONFIG['dataset_revision']
LOCAL_FILES_ONLY = DATA_CONFIG['local_files_only']
MAX_CALO_HITS = DIAGNOSTICS_CONFIG['max_calo_hits']
COORD_NOISE_SCALE = VIEW_CONFIG['coord_noise_scale']
ENERGY_JITTER_SCALE = VIEW_CONFIG['energy_jitter_scale']
GLOBAL_CROP_RATIO = VIEW_CONFIG['global_crop_ratio']
STUDENT_MASK_FRACTION = VIEW_CONFIG['student_mask_fraction']
POINT_DROPOUT = VIEW_CONFIG['point_dropout']
POINT_FEATURE_SAMPLE_SIZE = DIAGNOSTICS_CONFIG['point_feature_sample_size']
TOP_K_PROTOTYPES = DIAGNOSTICS_CONFIG['top_k_prototypes']
CHECKPOINT_PATH = PROJECT_ROOT / 'runs' / 'slurm_debug_train_small' / 'checkpoints' / 'best.pt'
METRICS_PATH = None
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESOLVED_CHECKPOINT_PATH = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else None

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Project root:', PROJECT_ROOT)
print('Using device:', DEVICE)
print('Dataset revision:', DATASET_REVISION)
print('Local files only:', LOCAL_FILES_ONLY)
print('Requested checkpoint:', CHECKPOINT_PATH)
print('Resolved checkpoint:', RESOLVED_CHECKPOINT_PATH)


In [ ]:
def pretty_json(payload: dict[str, Any]) -> None:
    print(json.dumps(payload, indent=2, sort_keys=True))


def infer_metrics_path(checkpoint_path: Path | None, metrics_path: Path | None) -> Path | None:
    if metrics_path is not None:
        return metrics_path
    if checkpoint_path is None:
        return None
    checkpoint = Path(checkpoint_path)
    run_dir = checkpoint.parent.parent if checkpoint.parent.name == 'checkpoints' else checkpoint.parent
    candidate = run_dir / 'metrics.jsonl'
    return candidate if candidate.exists() else None


def load_metric_records(metrics_path: Path | None) -> list[dict[str, Any]]:
    if metrics_path is None or not metrics_path.exists():
        return []
    return [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines() if line.strip()]


def metric_series(records: Sequence[dict[str, Any]], key: str) -> tuple[list[float], list[float]]:
    xs = []
    ys = []
    for index, record in enumerate(records, start=1):
        if key not in record:
            continue
        xs.append(float(record.get('epoch', index)))
        ys.append(float(record[key]))
    return xs, ys


def prototype_entropy(probabilities: torch.Tensor) -> torch.Tensor:
    probabilities = probabilities.clamp_min(1e-8)
    return -(probabilities * probabilities.log()).sum(dim=-1)


def assert_view_contract(view: dict[str, torch.Tensor]) -> dict[str, Any]:
    counts = torch.diff(view['offset'], prepend=view['offset'].new_zeros(1))
    return {
        'view_kind': view.get('view_kind', 'unknown'),
        'num_points': int(view['coord'].shape[0]),
        'points_per_event': counts.tolist(),
        'masked_points': int(view['mask'].sum().item()),
        'grid_size': float(view['grid_size'].item()),
        'all_values_finite': bool(torch.isfinite(view['feat']).all() and torch.isfinite(view['coord']).all()),
    }


def create_demo_model(device: torch.device, checkpoint_path: Path | None):
    if device.type != 'cuda':
        return None
    use_training_model = checkpoint_path is not None
    model = create_training_panda_model(device=device, **model_factory_kwargs(MODEL_CONFIG['training'])) if use_training_model else create_small_panda_model(device=device, **model_factory_kwargs(MODEL_CONFIG['diagnostics']))
    checkpoint_info = None
    if checkpoint_path is not None:
        checkpoint_info = load_checkpoint(model, str(checkpoint_path))
    model.eval()
    return model, checkpoint_info


## Start From A Real Event

Even when we're focused on the model, it helps to keep one concrete event in view so the later tensors don't feel abstract.


In [ ]:
detail_event = load_events(
    split=DETAIL_SPLIT,
    dataset_name=DATASET_NAME,
    batch_size=1,
    dataset_type=DATASET_TYPE,
    pu_config=PU_CONFIG,
    cache_dir=CACHE_DIR,
    dataset_revision=DATASET_REVISION,
    local_files_only=LOCAL_FILES_ONLY,
)[0]
representation_events = load_events(
    split=REPRESENTATION_SPLIT,
    dataset_name=DATASET_NAME,
    batch_size=10,
    dataset_type=DATASET_TYPE,
    pu_config=PU_CONFIG,
    cache_dir=CACHE_DIR,
    dataset_revision=DATASET_REVISION,
    local_files_only=LOCAL_FILES_ONLY,
)
training_events = load_events(
    split=f'train[:{TRAIN_VIEW_EVENT_COUNT}]',
    dataset_name=DATASET_NAME,
    batch_size=TRAIN_VIEW_EVENT_COUNT,
    dataset_type=DATASET_TYPE,
    pu_config=PU_CONFIG,
    cache_dir=CACHE_DIR,
    dataset_revision=DATASET_REVISION,
    local_files_only=LOCAL_FILES_ONLY,
)

print('Detailed event hit count:', len(detail_event['calo_hits']['x']))
print('Representation sample size:', len(representation_events))
print('Training event count for distillation batch:', len(training_events))


## Build The Canonical Point View

This is the first representation the model code actually cares about.

A `PointView` gives us geometry, model features, event boundaries, alignment back to the sampled raw hits, coarse patches for masking, and a human-readable label for diagnostics.


In [ ]:
base_view = build_point_view_from_event(
    detail_event,
    device=DEVICE,
    max_calo_hits=MAX_CALO_HITS,
)

pretty_json(assert_view_contract(base_view))
print('Feature tensor shape:', tuple(base_view['feat'].shape))
print('First five feature rows:')
print(base_view['feat'][:5])


In [ ]:
fig = plt.figure(figsize=(12, 5))
ax0 = fig.add_subplot(1, 2, 1, projection='3d')
coord = to_numpy(base_view['coord'])
energy = to_numpy(base_view['energy'])
scatter = ax0.scatter(coord[:, 2], coord[:, 0], coord[:, 1], c=energy, cmap='inferno', s=5, alpha=0.7)
fig.colorbar(scatter, ax=ax0, shrink=0.7, pad=0.1, label='energy')
ax0.set_title('Base point view geometry')
ax0.set_xlabel('z')
ax0.set_ylabel('x')
ax0.set_zlabel('y')

ax1 = fig.add_subplot(1, 2, 2)
ax1.hist(base_view['feat'][:, 3].cpu().numpy(), bins=40, color='tab:red', alpha=0.85)
ax1.set_title('Base point-view energy feature')
ax1.set_xlabel('energy')
plt.tight_layout()
plt.show()


## Augmentations We Use Right Now

The current augmentations are deliberately simple and readable.

What we use today:

- azimuthal rotation around the beam axis
- coordinate jitter
- energy jitter
- contiguous cropping
- point dropout
- coarse patch masking

This is still simpler than full Panda, but it already gives us a sensible masked-global SSL setup for ColliderML calorimeter hits.


In [ ]:
aug_teacher = augment_point_view(
    base_view,
    coord_noise_scale=COORD_NOISE_SCALE,
    feat_noise_scale=ENERGY_JITTER_SCALE,
    crop_keep_ratio=GLOBAL_CROP_RATIO,
    point_dropout=POINT_DROPOUT,
    mask_fraction=0.0,
    view_kind='teacher_demo',
)
aug_student = augment_point_view(
    base_view,
    coord_noise_scale=COORD_NOISE_SCALE,
    feat_noise_scale=ENERGY_JITTER_SCALE,
    crop_keep_ratio=GLOBAL_CROP_RATIO,
    point_dropout=POINT_DROPOUT,
    mask_fraction=STUDENT_MASK_FRACTION,
    view_kind='student_demo',
)

pretty_json({
    'teacher_demo': assert_view_contract(aug_teacher),
    'student_demo': assert_view_contract(aug_student),
})


In [ ]:
fig = plt.figure(figsize=(18, 6))
for index, (title, view) in enumerate([('base', base_view), ('teacher global', aug_teacher), ('student masked', aug_student)], start=1):
    ax = fig.add_subplot(1, 3, index, projection='3d')
    coord = to_numpy(view['coord'])
    energy = to_numpy(view['energy'])
    ax.scatter(coord[:, 2], coord[:, 0], coord[:, 1], c=energy, cmap='inferno', s=5, alpha=0.7)
    if view['mask'].any():
        masked_coord = to_numpy(view['coord'][view['mask']])
        ax.scatter(masked_coord[:, 2], masked_coord[:, 0], masked_coord[:, 1], color='cyan', s=12, alpha=0.8, label='masked')
        ax.legend(loc='upper right')
    ax.set_title(title)
    ax.set_xlabel('z')
    ax.set_ylabel('x')
    ax.set_zlabel('y')
plt.tight_layout()
plt.show()


## The Distillation Batch

The trainer does not feed single events directly into the model. It first builds a `DistillationBatch`:

- two batched teacher global views
- two batched student masked global views
- the per-event base views for reference and diagnostics

This is the object that `PandaSelfDistillation.forward()` actually consumes.


In [ ]:
distillation_batch = build_distillation_views(
    training_events,
    device=DEVICE,
    max_calo_hits=MAX_CALO_HITS,
    coord_noise_scale=COORD_NOISE_SCALE,
    feat_noise_scale=ENERGY_JITTER_SCALE,
    global_crop_ratio=GLOBAL_CROP_RATIO,
    student_mask_fraction=STUDENT_MASK_FRACTION,
    point_dropout=POINT_DROPOUT,
)

batch_report = {
    'base_views': [assert_view_contract(view) for view in distillation_batch['base_views']],
    'teacher_views': [assert_view_contract(view) for view in distillation_batch['teacher_views']],
    'student_views': [assert_view_contract(view) for view in distillation_batch['student_views']],
}
pretty_json(batch_report)


## Read The Model Source

The current model is still small enough that reading the implementation directly is worth it.


In [ ]:
print(inspect.getsource(PandaSelfDistillation))


## Model Pieces In Plain Language

The current `PandaSelfDistillation` model has a few main parts:

1. student and teacher backbones with the same architecture
2. student and teacher projectors
3. a student-only predictor
4. a shared prototype head
5. a running teacher center for stabilization

The teacher is not optimized directly. It tracks the student through an EMA update.


In [ ]:
model_bundle = create_demo_model(DEVICE, RESOLVED_CHECKPOINT_PATH)
if model_bundle is None:
    model = None
    checkpoint_info = None
    print('CUDA is unavailable. Model-backed sections will be explanatory only.')
else:
    model, checkpoint_info = model_bundle
    num_params = sum(parameter.numel() for parameter in model.parameters())
    print('Loaded model type:', type(model).__name__)
    print('Parameter count:', num_params)
    if checkpoint_info is not None:
        pretty_json(checkpoint_info)


In [ ]:
if model is not None:
    architecture_summary = {
        'student_backbone': type(model.student_backbone).__name__,
        'teacher_backbone': type(model.teacher_backbone).__name__,
        'student_projector': type(model.student_projector).__name__,
        'teacher_projector': type(model.teacher_projector).__name__,
        'student_predictor': type(model.student_predictor).__name__,
        'prototype_head_shape': list(model.prototype_head.weight.shape),
        'num_prototypes': int(model.num_prototypes),
        'temp_student': float(model.temp_student),
        'temp_teacher': float(model.temp_teacher),
        'center_shape': list(model.center.shape),
    }
    pretty_json(architecture_summary)


## Forward Pass And Output Tensors

The model gives us both point-level and pooled outputs.

That is important because the current objective mixes:

- point-level prototype matching on aligned points
- pooled masked-view matching at event level


In [ ]:
student_encoding = None
teacher_encoding = None
student_outputs = None
teacher_outputs = None
distillation_loss = None

if model is not None:
    with torch.no_grad():
        student_encoding = encode_view(model, aug_student, use_teacher=False)
        teacher_encoding = encode_view(model, aug_teacher, use_teacher=True)
        student_outputs, teacher_outputs = model(distillation_batch)
        distillation_loss = model.distillation_loss(student_outputs, teacher_outputs)

    forward_summary = {
        'student_encoding_shapes': {key: list(value.shape) for key, value in student_encoding.items() if isinstance(value, torch.Tensor)},
        'teacher_encoding_shapes': {key: list(value.shape) for key, value in teacher_encoding.items() if isinstance(value, torch.Tensor)},
        'num_student_views': len(student_outputs),
        'num_teacher_views': len(teacher_outputs),
        'distillation_loss': float(distillation_loss.item()),
    }
    pretty_json(forward_summary)


## Student And Teacher Agreement On One Event

A nice quick sanity plot is to compare the top prototype probabilities for the masked student summary and the corresponding teacher summary on the same event.

When training is working, this comparison should usually become more structured than pure random noise.


In [ ]:
if model is not None:
    student_probs = F.softmax(student_encoding['logits'][0], dim=-1)
    teacher_probs = F.softmax(teacher_encoding['logits'][0], dim=-1)
    combined = student_probs + teacher_probs
    top_indices = torch.topk(combined, k=min(TOP_K_PROTOTYPES, combined.shape[0])).indices.cpu().numpy()

    fig, ax = plt.subplots(figsize=(10, 4))
    positions = np.arange(len(top_indices))
    ax.bar(positions - 0.2, student_probs[top_indices].detach().cpu().numpy(), width=0.4, label='student', color='tab:blue')
    ax.bar(positions + 0.2, teacher_probs[top_indices].detach().cpu().numpy(), width=0.4, label='teacher', color='tab:orange')
    ax.set_xticks(positions)
    ax.set_xticklabels([str(int(index)) for index in top_indices])
    ax.set_title('Top prototype probabilities on one detailed event')
    ax.set_xlabel('prototype index')
    ax.set_ylabel('probability')
    ax.legend()
    plt.tight_layout()
    plt.show()


## How The Loss Is Put Together

The current loss has two pieces for every student/teacher pairing.

First, we align points by `source_index` and match the student and teacher prototype distributions point by point.

Second, we pool over masked points and also match a masked event-level summary.

That combination gives us something easy to explain and easy to debug, while still staying close to the masked-global SSL idea.


In [ ]:
print(inspect.getsource(pointwise_panda_loss))


## SSL Check: Random Init Versus Trained Checkpoint

One of the most useful comparisons early on is not "does the loss go down a bit?" but "does a trained checkpoint look different from the same architecture at random init?"

This is a quick way to tell whether training is at least moving the representation somewhere meaningful.


In [ ]:
comparison_metrics = None
random_encoding = None
trained_encoding = None
random_probs = None
trained_probs = None
representation_batch = None

if DEVICE.type == 'cuda':
    random_model = create_training_panda_model(device=DEVICE, **model_factory_kwargs(MODEL_CONFIG['training']))
    random_model.eval()

    trained_model = None
    if RESOLVED_CHECKPOINT_PATH is not None:
        trained_model = create_training_panda_model(device=DEVICE, **model_factory_kwargs(MODEL_CONFIG['training']))
        load_checkpoint(trained_model, str(RESOLVED_CHECKPOINT_PATH))
        trained_model.eval()

    representation_views = [
        build_point_view_from_event(event, device=DEVICE, max_calo_hits=MAX_CALO_HITS)
        for event in representation_events
    ]
    representation_batch = batch_point_views(representation_views)

    with torch.no_grad():
        random_encoding = encode_view(random_model, representation_batch, use_teacher=False)
        random_probs = F.softmax(random_encoding['logits'], dim=-1)

        if trained_model is not None:
            trained_encoding = encode_view(trained_model, representation_batch, use_teacher=False)
            trained_probs = F.softmax(trained_encoding['logits'], dim=-1)

    comparison_metrics = {
        'random_embedding_norm_mean': float(random_encoding['pooled'].norm(dim=1).mean().item()),
        'random_entropy_mean': float(prototype_entropy(random_probs).mean().item()),
    }
    if trained_model is not None:
        comparison_metrics |= {
            'trained_embedding_norm_mean': float(trained_encoding['pooled'].norm(dim=1).mean().item()),
            'trained_entropy_mean': float(prototype_entropy(trained_probs).mean().item()),
            'embedding_norm_delta': float(trained_encoding['pooled'].norm(dim=1).mean().item() - random_encoding['pooled'].norm(dim=1).mean().item()),
            'entropy_delta': float(prototype_entropy(trained_probs).mean().item() - prototype_entropy(random_probs).mean().item()),
        }

if comparison_metrics is not None:
    pretty_json(comparison_metrics)


In [ ]:
if DEVICE.type == 'cuda' and comparison_metrics is not None and trained_encoding is not None:
    labels = ['Embedding norm', 'Prototype entropy']
    random_values = [comparison_metrics['random_embedding_norm_mean'], comparison_metrics['random_entropy_mean']]
    trained_values = [comparison_metrics['trained_embedding_norm_mean'], comparison_metrics['trained_entropy_mean']]

    x = np.arange(len(labels))
    width = 0.36
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - width / 2, random_values, width, label='random init', color='tab:blue')
    ax.bar(x + width / 2, trained_values, width, label='trained checkpoint', color='tab:orange')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title('Random-init vs trained representation summary')
    ax.legend()
    plt.tight_layout()
    plt.show()


## SSL Check: Representation Geometry

PCA is not the final word on representation quality, but it is a fast way to see whether the feature space changes shape as training progresses.


In [ ]:
if DEVICE.type == 'cuda' and random_encoding is not None:
    active_encoding = trained_encoding if trained_encoding is not None else random_encoding
    pooled_pca = compute_pca(to_numpy(active_encoding['pooled']))
    point_indices = np.random.default_rng(SEED).choice(active_encoding['point_features'].shape[0], size=min(POINT_FEATURE_SAMPLE_SIZE, active_encoding['point_features'].shape[0]), replace=False)
    point_pca = compute_pca(to_numpy(active_encoding['point_features'][point_indices]))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(pooled_pca[:, 0], pooled_pca[:, 1], s=35, alpha=0.8, color='tab:blue')
    axes[0].set_title('Pooled embedding PCA')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')

    sampled_energy = to_numpy(representation_batch['energy'][point_indices])
    scatter = axes[1].scatter(point_pca[:, 0], point_pca[:, 1], c=sampled_energy, s=6, cmap='inferno', alpha=0.75)
    axes[1].set_title('Point-feature PCA')
    axes[1].set_xlabel('PC1')
    axes[1].set_ylabel('PC2')
    fig.colorbar(scatter, ax=axes[1], label='energy')
    plt.tight_layout()
    plt.show()


## Training Curves

A completed run gives us a few especially useful SSL signals in `metrics.jsonl`:

- train and validation loss
- prototype entropy
- embedding norm
- masked fraction
- learning-rate and teacher schedules
- center norm

Taken together, those plots usually tell us more than a single scalar about whether a run is healthy.


In [ ]:
metric_records = load_metric_records(infer_metrics_path(RESOLVED_CHECKPOINT_PATH, METRICS_PATH))
print('Metric record count:', len(metric_records))
if metric_records:
    pretty_json(metric_records[-1])


In [ ]:
if metric_records:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    train_x, train_y = metric_series(metric_records, 'train_loss')
    val_x, val_y = metric_series(metric_records, 'val_loss')
    axes[0, 0].plot(train_x, train_y, marker='o', label='train')
    axes[0, 0].plot(val_x, val_y, marker='o', label='val')
    axes[0, 0].set_title('Loss curves')
    axes[0, 0].legend()

    train_x, train_y = metric_series(metric_records, 'train_prototype_entropy')
    val_x, val_y = metric_series(metric_records, 'val_prototype_entropy')
    axes[0, 1].plot(train_x, train_y, marker='o', label='train')
    axes[0, 1].plot(val_x, val_y, marker='o', label='val')
    axes[0, 1].set_title('Prototype entropy')
    axes[0, 1].legend()

    train_x, train_y = metric_series(metric_records, 'train_embedding_norm')
    val_x, val_y = metric_series(metric_records, 'val_embedding_norm')
    axes[1, 0].plot(train_x, train_y, marker='o', label='train')
    axes[1, 0].plot(val_x, val_y, marker='o', label='val')
    axes[1, 0].set_title('Embedding norm')
    axes[1, 0].legend()

    lr_x, lr_y = metric_series(metric_records, 'learning_rate')
    tm_x, tm_y = metric_series(metric_records, 'teacher_momentum')
    tt_x, tt_y = metric_series(metric_records, 'teacher_temperature')
    axes[1, 1].plot(lr_x, lr_y, marker='o', label='learning rate')
    axes[1, 1].plot(tm_x, tm_y, marker='o', label='teacher momentum')
    axes[1, 1].plot(tt_x, tt_y, marker='o', label='teacher temperature')
    axes[1, 1].set_title('Schedules')
    axes[1, 1].legend(fontsize=8)

    for axis in axes.ravel():
        axis.set_xlabel('epoch')
    plt.tight_layout()
    plt.show()


## Run Health At A Glance

The gap between validation and training metrics is another quick read on whether a run is behaving sensibly.


In [ ]:
if metric_records:
    epochs = [float(record['epoch']) for record in metric_records]
    loss_gap = [float(record['val_loss']) - float(record['train_loss']) for record in metric_records]
    entropy_gap = [float(record['val_prototype_entropy']) - float(record['train_prototype_entropy']) for record in metric_records]
    norm_gap = [float(record['val_embedding_norm']) - float(record['train_embedding_norm']) for record in metric_records]
    masked_gap = [float(record['val_masked_fraction']) - float(record['train_masked_fraction']) for record in metric_records]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for axis, values, title in zip(
        axes.ravel(),
        [loss_gap, entropy_gap, norm_gap, masked_gap],
        ['Loss gap', 'Prototype entropy gap', 'Embedding norm gap', 'Masked fraction gap'],
    ):
        axis.plot(epochs, values, marker='o', color='tab:purple')
        axis.axhline(0.0, color='tab:gray', linestyle=':')
        axis.set_title(title)
        axis.set_xlabel('epoch')
    plt.tight_layout()
    plt.show()


## How The Trainer Fits In

The training script is intentionally explicit.

For each epoch it:

1. loads raw events from the dataloader
2. turns them into a `DistillationBatch`
3. runs student and teacher forward passes
4. computes the current distillation loss
5. updates the student with gradients
6. updates the teacher by EMA
7. logs metrics and saves checkpoints

Keeping those steps visible makes it much easier to debug than hiding them behind a lot of abstraction.


In [ ]:
from scripts.train import run_epoch
print(inspect.getsource(run_epoch))


## When Training Looks Off

A few questions are worth checking early, before spending time on long runs:

- are the view tensors finite and shape-consistent?
- is the masked fraction close to the configured target?
- are prototype entropies clearly non-zero?
- are embedding norms staying away from collapse?
- does a trained checkpoint look different from random init at all?
- are train and val losses at least in the same ballpark?
- are the teacher schedules doing what we think they are doing?

Those checks are easy to forget, but they usually save time.


## Practical Next Steps

Once this notebook makes sense, the normal next things to do are:

1. run `scripts/train.py` on a tiny split or via the SLURM jobs
2. inspect `runs/<run_name>/metrics.jsonl` and checkpoints
3. use `scripts/plot_training_run.py` for run-level curves
4. use `scripts/plot_diagnostics.py` for checkpoint-backed plots
5. compare multiple runs, not just one run against random init
